# Inicialización del entorno de trabajo

In [ ]:
import warnings
import pandas as pd
import pyreadr
from sklearn.impute import SimpleImputer
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
 )
from sklearn.metrics import roc_curve, precision_recall_curve
from IPython.display import Markdown, display
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from IPython.display import display
import time
import cProfile
import pstats
import io
from copy import deepcopy
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in reduce')

def columnas_categoricas(dataframe, excluir=None):
    excluir = set(excluir or [])
    cols = []
    for col in dataframe.columns:
        if col in excluir:
            continue
        dtype = dataframe[col].dtype
        if (
            pd.api.types.is_object_dtype(dtype)
            or pd.api.types.is_string_dtype(dtype)
            or isinstance(dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(dtype)
        ):
            cols.append(col)
    return cols

resultado = pyreadr.read_r('listings.RData')
df = list(resultado.values())[0]
print(f"Dimensiones iniciales del dataset: {df.shape}")

numericas = df.select_dtypes(include=['number']).columns.tolist()
categoricas = columnas_categoricas(df)

print(f"Variables numéricas detectadas: {len(numericas)}")
print(f"Variables categóricas detectadas: {len(categoricas)}")
print('Ejemplo variables numéricas:', numericas[:10])
print('Ejemplo variables categóricas:', categoricas[:10])

limpio = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
 )
df['price'] = pd.to_numeric(limpio, errors='coerce')
print('Conversión de price completada.')

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)

faltantes = df.isnull().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)

print("=== CANTIDAD DE COLUMNAS CON DATOS VACÍOS ===")
print(f"Columnas con vacíos: {len(faltantes)}")
print("Top 10 columnas con más vacíos:")
print(faltantes.head(10))
print("\n" + "=" * 40 + "\n")

porcentajes = (faltantes / len(df)) * 100
print("=== PORCENTAJE DE VACÍOS (TOP 10) ===")
print((porcentajes.head(10).round(2)).astype(str) + " %")

df = df.dropna(subset=['price'])
df = df.dropna(axis=1, how='all')

numericas = df.select_dtypes(include=['number']).columns.tolist()
if 'price' in numericas:
    numericas.remove('price')

categoricas = columnas_categoricas(df, excluir=['price'])

imputador_num = SimpleImputer(strategy='median')
imputador_cat = SimpleImputer(strategy='constant', fill_value='Sin Dato')

df[numericas] = imputador_num.fit_transform(df[numericas])
df[categoricas] = imputador_cat.fit_transform(df[categoricas])

print('Filas sin precio eliminadas y faltantes imputados.')
print('Total de nulos restantes:', df.isnull().sum().sum())
print('Dimensiones luego de limpieza:', df.shape)

=== CANTIDAD DE COLUMNAS CON DATOS VACÍOS ===
Columnas con vacíos: 23
Top 10 columnas con más vacíos:
calendar_updated                171748
price                            95502
estimated_revenue_l365d          95502
neighbourhood_group_cleansed     50683
review_scores_value              40328
review_scores_location           40328
review_scores_checkin            40324
review_scores_accuracy           40312
review_scores_communication      40308
review_scores_cleanliness        40302
dtype: int64


=== PORCENTAJE DE VACÍOS (TOP 10) ===
calendar_updated                100.0 %
price                           55.61 %
estimated_revenue_l365d         55.61 %
neighbourhood_group_cleansed    29.51 %
review_scores_value             23.48 %
review_scores_location          23.48 %
review_scores_checkin           23.48 %
review_scores_accuracy          23.47 %
review_scores_communication     23.47 %
review_scores_cleanliness       23.47 %
dtype: str
Filas sin precio eliminadas y faltantes impu

# \\\ Predicción de modelos de los precios de las casas //
#         - Empresa *SmartStay Advisors* -
# Maquinas Vectoriales de Soporte (SVM)

*Integrantes de la investigación:*
- Vianka Castro - 23201
- Ricardo Godinez - 23247
- Felipe Aguilar - 23195

Para esta nueva entrega hemos decidido proporcionarles un nuevo modelo operativo llamado Maquinas Vectoriales de Soporte para ayudarles a estimar los precios competitivos, las propiedades con baja ocupación, factores que influyen en la compra y diseño de estrategias basada en los datos para aumentar la rentabilidad de su empresa. 

Seguiremos utilizando el mismo dataset listings.RData que se nos fue proporcionado en la entrega anterior, el cual contiene información detallada sobre las propiedades listadas en la plataforma de SmartStay Advisors. Este dataset incluye variables como el precio, la ubicación, el número de habitaciones, la ocupación, entre otros factores relevantes para el análisis.

Por último les brindaremos una comparativa entre este modelo y los trabajados anteriormente para que puedan evaluar cuál de ellos se adapta mejor a sus necesidades y objetivos comerciales.

1. Comenzamos con la limpieza de datos y el uso de las mismas variables de entrenamiento y de prueba que fueron utilizadas en entregas anteriores. 

In [ ]:
# 1. ELIMINAR NULOS Y DEFINIR 'X' e 'Y'
# Filtramos las filas donde la categoría no sea nula
filas_sanas = df['categoria_precio'].notna()
df_limpio = df[filas_sanas].copy()

columnas_a_botar = [
    # Las que ya teníamos:
    'price', 'categoria_precio', 'es_economica', 'es_intermedia', 'es_cara',
    'id', 'scrape_id', 'host_id', 'last_scraped', 'calendar_last_scraped', 
    'first_review', 'last_review',
    
    # ---> NUEVAS A ELIMINAR POR COLINEALIDAD <---
    'latitude', 'longitude', # La ubicación ya la dará la variable 'city'
    'neighbourhood_group_cleansed', # Redundante con 'city'
    'host_verifications' # Arreglos de texto que dañan la matriz
]

X = df_limpio.drop(columns=[col for col in columnas_a_botar if col in df_limpio.columns])
X = X.replace([np.inf, -np.inf], np.nan)

columnas_falsas = [col for col in X.columns if col.endswith('_f') and len(X[col].unique()) <= 2]
X = X.drop(columns=columnas_falsas, errors='ignore')

# 2. Forzar explícitamente que las variables estructurales sean NUMÉRICAS
columnas_forzar_numero = ['accommodates', 'bathrooms', 'bedrooms', 'beds']
for col in columnas_forzar_numero:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')

# 3. Detección ESTRICTA de numéricas y categóricas
numericas = X.select_dtypes(include=['number']).columns.tolist()
categoricas = [col for col in X.columns if col not in numericas and X[col].nunique(dropna=True) < 50]
X[categoricas] = X[categoricas].astype(str)
# 'Y_strata' solo sirve para asegurar que la partición sea la misma de KNN
Y_strata = df_limpio['categoria_precio']

# 'Y_dicotomicas' almacena las tres columnas que vamos a predecir
Y_dicotomicas = df_limpio[['es_economica', 'es_intermedia', 'es_cara']]

# Detección de tipos para el preprocesador (igual al lab anterior)
numericas = X.select_dtypes(include=['number']).columns.tolist()
categoricas_crudas = X.select_dtypes(include=['object', 'category']).columns.tolist() 
categoricas = [col for col in categoricas_crudas if X[col].nunique(dropna=True) < 50]
X[categoricas] = X[categoricas].astype(str)

# 2. SEPARACIÓN DE DATOS DE ENTRENAMIENTO Y PRUEBA
X_train_clf, X_test_clf, Y_train_dico, Y_test_dico = train_test_split(
    X, Y_dicotomicas, test_size=0.3, random_state=42, stratify=Y_strata
)


preprocesador_clf = ColumnTransformer(
    transformers=[
        ('numeros', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())  
        ]), numericas),
        ('textos', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            
            ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
            
        ]), categoricas)
    ]
)

